# Unified Dataset Transformation Workflow

This notebook merges and cleans the workflows from `datasets-merging.ipynb` and `transformer-500k.ipynb`.

Goal: document how raw multi-source text datasets are transformed into age-labeled parquet datasets for the project (age-group text classification).


## What This Notebook Covers

1. Common setup and reusable helpers.
2. Reddit (ABCDE) TSV -> cleaned parquet + balanced subsets.
3. Twitter/X (ABCDE city + country) TSV -> cleaned parquet + merged dataset.
4. Blog Authorship CSV -> age-binned long vs short/medium parquet splits.
5. Hippocorpus CSV -> age-binned long-text parquet.
6. Final sanity checks and project-level summary.


In [ ]:
from pathlib import Path
import math

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.auto import tqdm

AGE_BINS = [6, 13, 18, 30, 50, 65, float("inf")]
AGE_LABELS = ["6-12", "13-17", "18-29", "30-49", "50-64", "65+"]
MODEL_CLASSES = ["13-17", "18-29", "30-49", "50-64", "65+"]
SEED = 0.42


In [ ]:
def first_existing_path(candidates):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these paths exists: {candidates}")


def ensure_parent_dir(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)


def age_distribution_parquet(parquet_path):
    dataset = ds.dataset(str(parquet_path), format="parquet")
    table = dataset.to_table(columns=["age"])
    vc = pc.value_counts(table["age"])

    counts = pd.DataFrame({
        "age": vc.field("values").to_pandas(),
        "count": vc.field("counts").to_pandas(),
    }).sort_values("count", ascending=False)
    counts["percent"] = (counts["count"] / counts["count"].sum() * 100).round(2)
    return counts.reset_index(drop=True)


def process_tsv_to_parquet(
    input_path,
    output_path,
    text_builder,
    age_col="DMGAgeAtPost",
    usecols=None,
    chunk_size=100_000,
):
    input_path = Path(input_path)
    output_path = Path(output_path)
    ensure_parent_dir(output_path)

    total_lines = sum(1 for _ in open(input_path, "r", encoding="utf-8")) - 1
    total_chunks = math.ceil(total_lines / chunk_size)

    writer = None
    reader = pd.read_csv(input_path, sep="	", chunksize=chunk_size, usecols=usecols)

    for chunk in tqdm(reader, total=total_chunks, desc=f"Processing {input_path.name}"):
        chunk["text"] = text_builder(chunk)
        age_num = pd.to_numeric(chunk[age_col], errors="coerce")
        chunk["age"] = pd.cut(age_num, bins=AGE_BINS, labels=AGE_LABELS, right=False)

        chunk = chunk[["text", "age"]].dropna(subset=["age"])
        chunk["text"] = chunk["text"].fillna("").astype(str)
        chunk["age"] = chunk["age"].astype(str)

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(str(output_path), table.schema, compression="snappy")
        writer.write_table(table)

    if writer is not None:
        writer.close()

    return output_path


## 1) Reddit (ABCDE): TSV -> cleaned parquet

In [ ]:
reddit_raw = first_existing_path([
    "data/abcde/reddit/reddit_users_posts.tsv",
    "data/reddit/reddit_users_posts.tsv",
])

reddit_processed = Path("data/abcde/reddit/other/reddit_processed.parquet")

reddit_processed = process_tsv_to_parquet(
    input_path=reddit_raw,
    output_path=reddit_processed,
    usecols=["PostTitle", "PostSelftext", "DMGAgeAtPost"],
    text_builder=lambda c: c["PostTitle"].fillna("").astype(str) + ". " + c["PostSelftext"].fillna("").astype(str),
    age_col="DMGAgeAtPost",
    chunk_size=100_000,
)

print("Saved:", reddit_processed)
age_distribution_parquet(reddit_processed)


## 2) Reddit balanced subsets (from transformer workflow)

In [ ]:
con = duckdb.connect(database=":memory:")
con.execute(f"SELECT setseed({SEED})")

reddit_100k_per_class = reddit_processed.parent / "reddit_processed_100k_per_class.parquet"
reddit_200k_per_class = reddit_processed.parent / "reddit_processed_200k_per_class.parquet"

for per_class, out_path in [(100_000, reddit_100k_per_class), (200_000, reddit_200k_per_class)]:
    query = f"""
    COPY (
      SELECT age, text
      FROM (
        SELECT
          age,
          text,
          ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
        FROM read_parquet('{reddit_processed}')
        WHERE age IN ({', '.join(f"'{c}'" for c in MODEL_CLASSES)})
          AND text IS NOT NULL
          AND length(text) > 0
      )
      WHERE rn <= {per_class}
    )
    TO '{out_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """
    con.execute(f"SELECT setseed({SEED})")
    con.execute(query)
    print("Saved:", out_path)

con.close()


In [ ]:
print("100k/class distribution")
age_distribution_parquet(reddit_100k_per_class)


## 3) Reddit learning-curve subsets (1k -> 500k)

In [ ]:
sizes = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000]
output_dir = Path("data/abcde/reddit")
output_dir.mkdir(parents=True, exist_ok=True)

max_per_class = max(sizes) // len(MODEL_CLASSES)

con = duckdb.connect(database=":memory:")
con.execute(f"SELECT setseed({SEED})")

con.execute(f"""
    CREATE TABLE reddit_pool AS
    SELECT text, age
    FROM (
        SELECT text, age,
               ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
        FROM read_parquet('{reddit_processed}')
        WHERE age IN ({', '.join(f"'{c}'" for c in MODEL_CLASSES)})
    )
    WHERE rn <= {max_per_class}
""")

for size in sizes:
    per_class = size // len(MODEL_CLASSES)
    con.execute(f"SELECT setseed({SEED})")
    df = con.execute(f"""
        SELECT text, age
        FROM (
            SELECT text, age,
                   ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
            FROM reddit_pool
        )
        WHERE rn <= {per_class}
        ORDER BY RANDOM()
    """).df()

    out_path = output_dir / f"reddit_{size // 1000}k.parquet"
    df.to_parquet(out_path, index=False)
    print(f"Saved {out_path} ({len(df):,} rows)")

con.close()


## 4) Twitter/X (ABCDE): city + country TSV -> cleaned parquet -> merged parquet

In [ ]:
twitter_city_raw = first_existing_path([
    "data/abcde/twitter/other/city_user_posts.tsv",
    "data/abcde/tusc/city_user_posts.tsv",
])

twitter_country_raw = first_existing_path([
    "data/abcde/twitter/other/country_user_posts.tsv",
    "data/abcde/tusc/country_user_posts.tsv",
])

twitter_city_parquet = Path("data/abcde/twitter/other/tusc_processed.parquet")
twitter_country_parquet = Path("data/abcde/twitter/other/country_user_posts.parquet")
twitter_merged_parquet = Path("data/abcde/twitter/other/tusc_merged.parquet")

process_tsv_to_parquet(
    input_path=twitter_city_raw,
    output_path=twitter_city_parquet,
    usecols=["PostText", "DMGAgeAtPost"],
    text_builder=lambda c: c["PostText"],
    age_col="DMGAgeAtPost",
    chunk_size=100_000,
)

process_tsv_to_parquet(
    input_path=twitter_country_raw,
    output_path=twitter_country_parquet,
    usecols=["PostText", "DMGAgeAtPost"],
    text_builder=lambda c: c["PostText"],
    age_col="DMGAgeAtPost",
    chunk_size=100_000,
)

twitter_city_df = pd.read_parquet(twitter_city_parquet)
twitter_country_df = pd.read_parquet(twitter_country_parquet)

twitter_merged_df = pd.concat([twitter_city_df, twitter_country_df], ignore_index=True)
twitter_merged_df.to_parquet(twitter_merged_parquet, index=False)

print("Saved:", twitter_merged_parquet)
age_distribution_parquet(twitter_merged_parquet)


In [ ]:
con = duckdb.connect(database=":memory:")
con.execute(f"SELECT setseed({SEED})")

count_13_17 = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{twitter_merged_parquet}') WHERE age = '13-17'
""").fetchone()[0]

per_other = max((500_000 - count_13_17) // 4, 0)
twitter_500k = Path("data/abcde/twitter/twitter_500k.parquet")

query = f"""
COPY (
    SELECT text, age
    FROM (
        SELECT text, age,
               ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
        FROM read_parquet('{twitter_merged_parquet}')
        WHERE age IN ({', '.join(f"'{c}'" for c in MODEL_CLASSES)})
    )
    WHERE (age = '13-17')
       OR (age != '13-17' AND rn <= {per_other})
    ORDER BY RANDOM()
)
TO '{twitter_500k}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)
con.close()

print("Saved:", twitter_500k)
age_distribution_parquet(twitter_500k)


## 5) Blog Authorship: CSV -> age-binned long vs short/medium parquet

In [ ]:
blog_csv = first_existing_path(["blogtext.csv", "data/blog/blogtext.csv"])
blog_df = pd.read_csv(blog_csv)

blog_df["text"] = blog_df["text"].fillna("").astype(str)
blog_df["len_chars"] = blog_df["text"].str.len()
blog_df["age"] = pd.cut(pd.to_numeric(blog_df["age"], errors="coerce"), bins=AGE_BINS, labels=AGE_LABELS, right=False)

threshold = blog_df["len_chars"].quantile(0.75)
blog_long_df = blog_df.loc[blog_df["len_chars"] > threshold, ["text", "age"]].dropna(subset=["age"]).copy()
blog_short_medium_df = blog_df.loc[blog_df["len_chars"] <= threshold, ["text", "age"]].dropna(subset=["age"]).copy()

blog_long_df["age"] = blog_long_df["age"].astype(str)
blog_short_medium_df["age"] = blog_short_medium_df["age"].astype(str)

blog_long_path = Path("data/blog/blog_long.parquet")
blog_short_medium_path = Path("data/blog/blog_short_medium.parquet")
ensure_parent_dir(blog_long_path)

blog_long_df.to_parquet(blog_long_path, index=False)
blog_short_medium_df.to_parquet(blog_short_medium_path, index=False)

print("Saved:", blog_long_path)
print("Saved:", blog_short_medium_path)
print("Threshold (75th percentile):", round(threshold, 2))


## 6) Hippocorpus: CSV -> age-binned long-text parquet

In [ ]:
hippo_csv = first_existing_path(["hippoCorpusV2.csv", "data/hippocorpus/hippoCorpusV2.csv"])
hippo_df = pd.read_csv(hippo_csv, usecols=["story", "annotatorAge"])
hippo_df = hippo_df.rename(columns={"story": "text", "annotatorAge": "age"})

hippo_df["age"] = pd.cut(pd.to_numeric(hippo_df["age"], errors="coerce"), bins=AGE_BINS, labels=AGE_LABELS, right=False)
hippo_df = hippo_df[["text", "age"]].dropna(subset=["age"]).copy()
hippo_df["text"] = hippo_df["text"].fillna("").astype(str)
hippo_df["age"] = hippo_df["age"].astype(str)

hippo_df["token_len"] = hippo_df["text"].str.split().map(len)
hippo_long_df = hippo_df.loc[hippo_df["token_len"] > 128, ["text", "age"]]

hippo_long_path = Path("data/hippocorpus/hippocorpus_long.parquet")
ensure_parent_dir(hippo_long_path)
hippo_long_df.to_parquet(hippo_long_path, index=False)

print("Saved:", hippo_long_path)
age_distribution_parquet(hippo_long_path)


## 7) Final summary table for transformed outputs

In [ ]:
final_outputs = {
    "reddit_processed": Path("data/abcde/reddit/other/reddit_processed.parquet"),
    "reddit_100k_per_class": Path("data/abcde/reddit/other/reddit_processed_100k_per_class.parquet"),
    "reddit_200k_per_class": Path("data/abcde/reddit/other/reddit_processed_200k_per_class.parquet"),
    "twitter_merged": Path("data/abcde/twitter/other/tusc_merged.parquet"),
    "twitter_500k": Path("data/abcde/twitter/twitter_500k.parquet"),
    "blog_long": Path("data/blog/blog_long.parquet"),
    "blog_short_medium": Path("data/blog/blog_short_medium.parquet"),
    "hippocorpus_long": Path("data/hippocorpus/hippocorpus_long.parquet"),
}

rows = []
for name, path in final_outputs.items():
    if path.exists():
        try:
            row_count = duckdb.query(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
            rows.append({"dataset": name, "path": str(path), "rows": row_count})
        except Exception as e:
            rows.append({"dataset": name, "path": str(path), "rows": f"error: {e}"})
    else:
        rows.append({"dataset": name, "path": str(path), "rows": "missing"})

pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)


## Notes on Corrections Applied

- Removed notebook-only `pip install ...` cell from the workflow.
- Standardized path handling and added fallbacks for legacy folder layouts.
- Added helper functions to avoid repeated copy-paste transformations.
- Added explicit parent-directory creation before writing parquet outputs.
- Kept deterministic sampling with a fixed random seed.
- Added final sanity checks so the notebook clearly demonstrates project-ready transformed datasets.
